# Job Finder Chatbot — MLflow Evaluation
ISA632 Group Project | Evaluation & Validation

Uses `mlflow.genai.evaluate()` (MLflow 3.0) with custom Guidelines scorers and built-in scorers
applied to 12 realistic user questions from our ChatTesting session.

In [0]:
%pip install -qqq -U databricks-sdk databricks-langchain databricks-vectorsearch langsmith>=0.1.125 langchain==0.3.27 mlflow-skinny[databricks]>=3.4.0 
dbutils.library.restartPython()

In [0]:
# Cell 1 — Setup
import os
import json
import mlflow
import mlflow.pyfunc
import pandas as pd
from mlflow.genai import evaluate
from mlflow.genai.scorers import Guidelines, RelevanceToQuery, Safety

os.environ["VS_INDEX_NAME"] = "isa632_7474656346303369.boopatt.jobindex"
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment("/Users/boopatt@miamioh.edu/JobFinder_Eval")
print("Setup complete.")

In [0]:
# Cell 2 — Load the Registered Model
model_uri = "models:/isa632_7474656346303369.boopatt.getstarted_job_listings/1"
model = mlflow.pyfunc.load_model(model_uri)
print(f"Model loaded: {model_uri}")

In [0]:
# Cell 3 — Define predict_fn
@mlflow.trace
def predict_fn(messages):
    """Wraps the job finder agent for MLflow evaluation."""
    out = model.predict({"input": messages})

    if isinstance(out, dict) and "output" in out:
        try:
            return out["output"][-1]["content"][0]["text"].strip()
        except Exception:
            pass
    if isinstance(out, list):
        try:
            return out[-1]["content"][0]["text"].strip()
        except Exception:
            pass
    if isinstance(out, str):
        return out.strip()
    return str(out).strip()

# Quick smoke test
test_response = predict_fn([{"content": "What data analyst roles are available?", "role": "user"}])
print("Smoke test response (first 200 chars):")
print(test_response[:200])

In [0]:
# Cell 4 — Load Evaluation Dataset
eval_rows = []
with open("eval_data.jsonl") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        record = json.loads(line)
        eval_rows.append({
            "inputs": record["inputs"],
            "expectations": record["expectations"]
        })

eval_df = pd.DataFrame(eval_rows)
print(f"Loaded {len(eval_df)} evaluation questions.")
display(eval_df)

In [0]:
# Cell 5 — Define Custom Guidelines Scorers
job_relevant_scorer = Guidelines(
    name="job_relevant",
    guidelines="""The response must reference real job postings from the database.
It should include at least one of: a job title, company name, job description, salary range, or qualification requirement.
If the question is about images or unrelated topics (e.g., asking for a photo), the response should clearly state it cannot help — that is also acceptable.
Responses that are vague, generic, or fabricate job information without citing the database are NOT acceptable."""
)

filter_aware_scorer = Guidelines(
    name="filter_aware",
    guidelines="""When the user specifies a filter — such as a location (e.g., New York), experience level (e.g., beginner), 
skill requirement (e.g., Python only), or work mode (e.g., remote) — the response must acknowledge that constraint explicitly.
If no matching results exist for the filter, the response must say so rather than returning unfiltered results silently.
If the question contains no filter constraints, this criterion does not apply — score as passing."""
)

print("Custom scorers defined: job_relevant, filter_aware")

In [0]:
# Cell 6 — Run Evaluation
with mlflow.start_run(run_name="jobfinder_eval_v1") as eval_run:
    eval_results = evaluate(
        data=eval_df,
        predict_fn=predict_fn,
        scorers=[
            job_relevant_scorer,
            filter_aware_scorer,
            RelevanceToQuery(),
            Safety()
        ]
    )

print(f"Evaluation complete. Run ID: {eval_run.info.run_id}")

In [0]:
# Cell 7 — Summarize Results
results_df = eval_results.tables["eval_results"]

score_cols = [
    "job_relevant/value",
    "filter_aware/value",
    "relevance_to_query/value",
    "safety/value"
]

def to_numeric(val):
    """Convert boolean/string scorer values to 0 or 1."""
    if isinstance(val, bool):
        return int(val)
    if isinstance(val, str):
        return 1 if val.lower() in ("yes", "true") else 0
    return int(bool(val))

print("=== Evaluation Summary ===")
print(f"Total questions evaluated: {len(results_df)}")
print()
for col in score_cols:
    if col in results_df.columns:
        mean_val = results_df[col].apply(to_numeric).mean()
        label = col.replace("/value", "").replace("_", " ").title()
        print(f"{label:30s}: {mean_val:.0%}")

print()
print("View full results in MLflow UI \u2192 Experiments \u2192 JobFinder_Eval")